# Hybrid OKF: Qdrant vectors + OKF bundle + links

This is the OKF counterpart of `06_graph_query_hybrid.ipynb`. Together, the four query notebooks form a 2×2 comparison:

| | keyword only (BM25) | hybrid (vectors + BM25 + RRF) |
|---|---|---|
| **OKF** (curated wiki + markdown links) | `02_OKF_query.ipynb` | **this notebook** |
| **Knowledge graph** (facts + communities) | `04_graph_query.ipynb` | `06_graph_query_hybrid.ipynb` |

Each piece mirrors one in the graph hybrid:

| `06_graph_query_hybrid` | here |
|---|---|
| Qdrant search over `okf_chunks` | **the same search** over `okf_chunks`. Each chunk found is mapped to the **OKF section** it sits in, so the model reads whole sections like `02_OKF_query` does. |
| graph walk from the question's entities | **following OKF links**: topic page → the sections it cites |
| vector → graph (facts from the chunks found) | **vector → wiki**: the sections found → the topic pages that cite them (backlinks) |
| community reports in Qdrant (`okf_communities`) | **topic pages** in Qdrant (`okf_topics`, uploaded in step 4) |
| BM25 over facts / chunks | BM25 over concepts (same as `02_OKF_query`) |
| RRF, question splitting, 18,000-char budget, same model and questions | the same |

Because both hybrids use **the same vector search**, any difference between them comes from what's on top of it: a curated wiki, or an extracted graph.

```
question ──► split two-part questions
   ├─► A. Qdrant okf_chunks (each part)  → chunks → their OKF sections      → ranked section list(s)
   ├─► B. BM25 over the bundle                                               → ranked sections, ranked topic pages
   ├─► C. topic pages: Qdrant okf_topics + BM25 + backlinks from A's sections → RRF → 2 topic pages
   ├─► D. links: sections cited by those topic pages                         → ranked section list
   ├─► E. fuse A + B + D with RRF                                            → the sections
   └─► context = topic pages + sections → qwen3-4b → answer citing [concept-id]
```

## 1. Setup

In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the project root, wherever Jupyter starts
    os.chdir(Path.cwd().parent)

import hashlib
import json
import math
import os
import re
import time
import uuid
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import requests
import yaml
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BUNDLE_DIR = PROJECT_ROOT / "OKF_Bundle"
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"

# LLM: same as the other three notebooks
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b-instruct-2507-q4_K_M"
NUM_CTX = 8192
MAX_ANSWER_TOKENS = 700
CONTEXT_CHARS = 18000
MAX_CHARS_PER_CONCEPT = 4500        # same as OKF_query

# Qdrant: okf_chunks from 01_Chunking.ipynb, plus okf_topics made in step 4
CHUNK_COLLECTION = "okf_chunks"
TOPIC_COLLECTION = "okf_topics"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "qdrant/bm25"
LATE_INTERACTION_MODEL = "answerdotai/answerai-colbert-small-v1"
PREFETCH_LIMIT = 30

# switches
USE_VECTORS = True       # Qdrant chunk search -> sections
USE_LINKS = True         # topic page -> cited sections, and section -> citing topic pages
SPLIT_QUESTIONS = True

# knobs
N_VECTOR = 10            # chunks from Qdrant per query
N_TOPICS = 2             # topic pages in the context
N_SECTIONS = 5           # sections in the context (they're long, so the budget usually stops at 3-4)
PER_PAPER = 2            # max sections per paper, unless the question names a paper
PAPER_BOOST = 1.5
RRF_K = 60

assert (BUNDLE_DIR / "index.md").exists(), f"OKF bundle not found at {BUNDLE_DIR}"
print("project:", PROJECT_ROOT)

project: c:\Users\Swapn\OneDrive\Documents\GitHub\OKF


In [2]:
# connect to Qdrant Cloud (credentials from .env, like 01_Chunking.ipynb)
load_dotenv(PROJECT_ROOT / ".env")
QDRANT_OK = False
try:
    client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"), cloud_inference=True, timeout=60)
    print(f"{CHUNK_COLLECTION}: {client.count(CHUNK_COLLECTION, exact=True).count} points")
    QDRANT_OK = True
except Exception as exc:
    print(f"!! Qdrant not reachable ({exc}).\n!! Vector search is OFF - this run is OKF + BM25 only.")
USE_VECTORS = USE_VECTORS and QDRANT_OK

# check: Ollama
tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=10).json()
assert MODEL_NAME in [m["name"] for m in tags.get("models", [])], f"ollama pull {MODEL_NAME}"

okf_chunks: 2927 points


## 2. Load the bundle and its links (same as `02_OKF_query`)

One file = one concept. The concept ID is the path without `.md`. Markdown links between files are the edges:
- `out_links[A]`: what A points to. A topic page points to the sections it cites.
- `in_links[A]`: what points to A. For a section, these are the topic pages that cite it.

In [3]:
RESERVED = {"index.md", "log.md"}
LINK_RE = re.compile(r"\[([^\]]*)\]\((/[^)#\s]+)\.md\)")


def load_bundle(bundle_dir):
    concepts = {}
    for path in sorted(bundle_dir.rglob("*.md")):
        if path.name in RESERVED:
            continue
        _, frontmatter, body = path.read_text(encoding="utf-8").split("---\n", 2)
        fm = yaml.safe_load(frontmatter)
        cid = path.relative_to(bundle_dir).with_suffix("").as_posix()
        concepts[cid] = {"id": cid, "type": fm["type"], "title": fm.get("title", cid),
                         "description": fm.get("description", ""), "tags": fm.get("tags") or [],
                         "aliases": fm.get("aliases") or [], "paper": fm.get("paper"),
                         "pages": fm.get("pages"), "body": body}
    return concepts


concepts = load_bundle(BUNDLE_DIR)
out_links, in_links = defaultdict(set), defaultdict(set)
for cid, c in concepts.items():
    for _, target in LINK_RE.findall(c["body"]):
        t = target.lstrip("/")
        if t in concepts and t != cid:
            out_links[cid].add(t)
            in_links[t].add(cid)


def is_wiki(cid):
    return concepts[cid]["type"] != "Paper Section"


wiki_ids = [cid for cid in concepts if is_wiki(cid)]
print(f"concepts: {len(concepts)} ({len(wiki_ids)} topic/overview pages, {len(concepts) - len(wiki_ids)} sections) | "
      f"links: {sum(len(v) for v in out_links.values())}")

concepts: 857 (59 topic/overview pages, 798 sections) | links: 4408


## 3. Map every chunk to its OKF section

Qdrant returns **chunks** (~1,000 characters, from `01_Chunking.ipynb`). The bundle's unit is the **section**. Both were cut from the same cleaned markdown, so a chunk's text appears inside exactly one section. We find it by looking up a short piece of the chunk's text in the sections of the same paper.

After this, a Qdrant hit `Holistic_Eval_0126` becomes the section `papers/helm/4-4-calibration-and-uncertainty`.

In [4]:
DOC_TO_PAPER = {"AGI": "sparks-of-agi", "GPT-4": "gpt-4-technical-report", "Holistic_Eval": "helm",
                "Foundation_Models": "foundation-models", "LLMs": "llm-survey"}


def normalize(text):
    text = re.sub(r"\[p\. \d+\]", " ", text)                  # page markers exist only in the bundle
    text = re.sub(r"[*_`#|>\-–—]+", " ", text)                  # markdown that differs between the two
    return re.sub(r"\s+", " ", text).strip().lower()


section_text = {cid: normalize(c["body"].split("\n## Links")[0]) for cid, c in concepts.items() if not is_wiki(cid)}
sections_of_paper = defaultdict(list)
for cid in section_text:
    sections_of_paper[cid.split("/")[1]].append(cid)

with CHUNKS_PATH.open(encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

section_of_chunk = {}
for c in chunks:
    text = normalize(c["text"])
    for start in (0, len(text) // 3, 2 * len(text) // 3):         # try 3 probes from different places
        probe = text[start:start + 60]
        if len(probe) < 20:
            continue
        hits = [cid for cid in sections_of_paper[DOC_TO_PAPER[c["document"]]] if probe in section_text[cid]]
        if hits:
            section_of_chunk[c["chunk_id"]] = min(hits, key=lambda h: len(section_text[h]))
            break

missed = [c for c in chunks if c["chunk_id"] not in section_of_chunk]
print(f"mapped {len(section_of_chunk)} of {len(chunks)} chunks to sections")
print("not mapped (front matter, acknowledgments - the bundle skips those):", Counter(c["document"] for c in missed))

mapped 2900 of 2927 chunks to sections
not mapped (front matter, acknowledgments - the bundle skips those): Counter({'Holistic_Eval': 16, 'Foundation_Models': 6, 'GPT-4': 2, 'LLMs': 2, 'AGI': 1})


## 4. Topic pages in Qdrant (`okf_topics`)

This mirrors `okf_communities`: the curated topic pages and paper overviews (~59) go into their own small collection, with the same dense + BM25 + ColBERT setup.

What gets embedded is the page's **title, description, aliases and summary**. MiniLM only reads the first ~256 tokens of a text, so this is the part that says what the page is about.

The upload uses a content hash, so re-running this cell only uploads pages that changed. `okf_chunks` is not touched.

In [5]:
def topic_text(cid):
    c = concepts[cid]
    body = LINK_RE.sub(r"\1", c["body"])
    summary = body.split("## Summary", 1)[-1].split("\n## ", 1)[0].strip() if "## Summary" in body else body[:1500]
    return f"{c['title']}\n{c['description']}\nAlso called: {', '.join(c['aliases'])}\n\n{summary}"[:2000]


def topic_point_id(cid):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"okf_topic:{cid}"))


TOPICS_IN_QDRANT = False
if QDRANT_OK:
    if not client.collection_exists(TOPIC_COLLECTION):
        client.create_collection(
            TOPIC_COLLECTION,
            vectors_config={
                "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
                "multi": models.VectorParams(size=96, distance=models.Distance.COSINE,
                                             multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM),
                                             hnsw_config=models.HnswConfigDiff(m=0)),
            },
            sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
        )
        print(f"created collection '{TOPIC_COLLECTION}'")
    stored, offset = {}, None
    while True:
        points, offset = client.scroll(TOPIC_COLLECTION, limit=256, offset=offset, with_payload=["content_hash"], with_vectors=False)
        stored.update({str(p.id): (p.payload or {}).get("content_hash") for p in points})
        if offset is None:
            break
    wanted = {topic_point_id(cid): cid for cid in wiki_ids}
    todo = [cid for pid, cid in wanted.items() if stored.get(pid) != hashlib.sha256(topic_text(cid).encode()).hexdigest()]
    stale = [pid for pid in stored if pid not in wanted]
    if stale:
        client.delete(TOPIC_COLLECTION, points_selector=models.PointIdsList(points=stale))
    for i in range(0, len(todo), 32):
        client.upsert(TOPIC_COLLECTION, wait=True, points=[
            models.PointStruct(
                id=topic_point_id(cid),
                vector={"dense": models.Document(text=topic_text(cid), model=DENSE_MODEL),
                        "sparse": models.Document(text=topic_text(cid), model=SPARSE_MODEL),
                        "multi": models.Document(text=topic_text(cid), model=LATE_INTERACTION_MODEL)},
                payload={"concept_id": cid, "title": concepts[cid]["title"], "type": concepts[cid]["type"],
                         "content_hash": hashlib.sha256(topic_text(cid).encode()).hexdigest()},
            ) for cid in todo[i:i + 32]])
    TOPICS_IN_QDRANT = True
    print(f"{TOPIC_COLLECTION}: {client.count(TOPIC_COLLECTION, exact=True).count} points "
          f"(uploaded {len(todo)}, unchanged {len(wanted) - len(todo)}, deleted {len(stale)})")

created collection 'okf_topics'
okf_topics: 59 points (uploaded 59, unchanged 0, deleted 0)


## 5. Search tools: BM25, Qdrant, question splitting, RRF

- **BM25** is the same field-weighted search as `02_OKF_query`: title ×3, aliases/tags/description ×2.
- **Qdrant** runs the same query as `01_Chunking.ipynb`: dense + sparse prefetch, then ColBERT rerank.
- **Question splitting** is the same rule as the graph hybrid: split at ", and how/what/why…" only if both halves make sense on their own. The graph hybrid checks for an entity name; here each half must name something the wiki has a page for (a title or alias, like "RLHF" or "calibration").
- **RRF** is the same: `1 / (60 + rank)` summed over the lists.

In [6]:
STOPWORDS = set('''a an the of and or to in on for with by from at as is are was were be been being it its this that these
those what which who whom how why when where do does did can could should would will may might about into than then
there their they them we our you your i me my he she his her not no yes vs'''.split())


def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+(?:[-.][a-z0-9]+)*", text.lower()) if t not in STOPWORDS]


class BM25:
    def __init__(self, concepts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.tf, self.length = {}, {}
        for cid, c in concepts.items():
            body = LINK_RE.sub(r"\1", c["body"].split("\n## Links")[0])
            fields = ((c["title"] + " ") * 3 + (" ".join(c["aliases"] + c["tags"]) + " " + c["description"] + " ") * 2 + body)
            tokens = tokenize(fields)
            self.tf[cid], self.length[cid] = Counter(tokens), len(tokens)
        self.avg_len = sum(self.length.values()) / len(self.length)
        df = Counter(w for tf in self.tf.values() for w in tf)
        n = len(self.tf)
        self.idf = {w: math.log(1 + (n - d + 0.5) / (d + 0.5)) for w, d in df.items()}

    def scores(self, query, keys=None):
        q = tokenize(query)
        result = {}
        for cid in (self.tf if keys is None else keys):
            tf = self.tf[cid]
            norm = 1 - self.b + self.b * self.length[cid] / self.avg_len
            s = sum(self.idf[w] * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * norm) for w in q if w in tf)
            if s > 0:
                result[cid] = s
        return result


index = BM25(concepts)


def qdrant_search(collection, query, top_k, id_field):
    result = client.query_points(
        collection,
        prefetch=[models.Prefetch(query=models.Document(text=query, model=DENSE_MODEL), using="dense", limit=PREFETCH_LIMIT),
                  models.Prefetch(query=models.Document(text=query, model=SPARSE_MODEL), using="sparse", limit=PREFETCH_LIMIT)],
        query=models.Document(text=query, model=LATE_INTERACTION_MODEL),
        using="multi", limit=top_k, with_payload=[id_field],
    )
    return [p.payload[id_field] for p in result.points]


SPLIT_RE = re.compile(r",?\s+and\s+(?=(?:how|what|why|which|who|when|where|does|did|is|are|was|were)\b)", re.IGNORECASE)


# a half "makes sense on its own" if it names something the wiki has a page for (title or alias),
# like the graph hybrid's rule "each half names an entity"
TOPIC_NAMES = sorted({n.lower() for cid in wiki_ids for n in [concepts[cid]["title"]] + concepts[cid]["aliases"]
                      if len(n) >= 3 and not cid.endswith("/overview")}, key=len, reverse=True)


def names_a_topic(text):
    text = text.lower()
    return any(re.search(r"(?<![a-z0-9])" + re.escape(n) + r"(?![a-z0-9])", text) for n in TOPIC_NAMES)


def split_question(question):
    parts = [p.strip(" ?,") for p in SPLIT_RE.split(question)]
    if len(parts) > 1 and all(names_a_topic(p) for p in parts):
        return parts
    return []


def rrf(*ranked_lists, k=RRF_K):
    score = Counter()
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked, start=1):
            score[item] += 1 / (k + rank)
    return score


PAPER_HINTS = {
    "sparks-of-agi": ["sparks of agi", "sparks of artificial general intelligence", "sparks paper"],
    "gpt-4-technical-report": ["gpt-4 technical report", "gpt-4 report", "technical report", "system card"],
    "helm": ["helm", "holistic evaluation of language models"],
    "foundation-models": ["foundation models report", "opportunities and risks of foundation models"],
    "llm-survey": ["llm survey", "survey of large language models"],
}


def papers_named(question):
    q = question.lower()
    return [p for p, hints in PAPER_HINTS.items() if any(re.search(r"(?<![a-z0-9])" + re.escape(h) + r"(?![a-z0-9])", q) for h in hints)]


def paper_of(cid):
    return cid.split("/")[1] if cid.startswith("papers/") else None


for q in ["What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?",
          "Which papers discuss hallucination, and what does each one say about it?"]:
    print(f"{q}\n   -> {split_question(q) or '(not split)'}")

What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?
   -> ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
Which papers discuss hallucination, and what does each one say about it?
   -> (not split)


## 6. Retrieve

1. **Vector:** Qdrant chunks for the question and each part → mapped to sections. One ranked section list per query; a section's rank is its best chunk's rank.
2. **BM25:** sections and topic pages ranked by keywords (the best score over the question and its parts).
3. **Topic pages:** RRF of Qdrant `okf_topics` search, BM25, and **backlinks**: the topic pages that cite the sections from step 1, most cited first. This is "vector → wiki". Keep the top `N_TOPICS`.
4. **Links:** the sections those topic pages cite, ordered by BM25. This is "wiki → evidence".
5. **Fuse** sections with RRF: vector lists + BM25 + links. A named paper gets `PAPER_BOOST`, otherwise at most `PER_PAPER` sections per paper.

Each section records **who found it** (`vector`, `bm25`, `link`).

In [7]:
def retrieve(question, use_vectors=None, use_links=None, split=None, use_topics=True):
    use_vectors = USE_VECTORS if use_vectors is None else (use_vectors and QDRANT_OK)
    use_links = USE_LINKS if use_links is None else use_links
    split = SPLIT_QUESTIONS if split is None else split
    timing = {}
    papers = papers_named(question)
    parts = split_question(question) if split else []
    queries = [question] + parts

    # 1) vector: chunks -> sections
    t = time.perf_counter()
    vector_lists = []
    if use_vectors:
        for qx in queries:
            mapped = [section_of_chunk[c] for c in qdrant_search(CHUNK_COLLECTION, qx, N_VECTOR, "chunk_id") if c in section_of_chunk]
            vector_lists.append(list(dict.fromkeys(mapped)))
    timing["vector_ms"] = (time.perf_counter() - t) * 1000

    # 2) BM25 over the bundle
    t = time.perf_counter()
    bm25 = Counter()
    for qx in queries:
        for cid, s in index.scores(qx).items():
            bm25[cid] = max(bm25[cid], s)
    bm25_sections = [cid for cid, _ in bm25.most_common() if not is_wiki(cid)][:30]
    bm25_topics = [cid for cid, _ in bm25.most_common() if is_wiki(cid)][:10]

    # 3) topic pages: Qdrant + BM25 + backlinks from the vector sections
    topics = []
    if use_topics:
        searched = [qdrant_search(TOPIC_COLLECTION, qx, 5, "concept_id") for qx in queries] if (use_vectors and TOPICS_IN_QDRANT) else []
        vector_sections = [c for lst in vector_lists for c in lst[:5]]
        backlinks = Counter(tp for c in vector_sections for tp in in_links[c] if is_wiki(tp)) if use_links else Counter()
        backlink_list = [t for t, _ in backlinks.most_common()]
        fused_topics = rrf(*searched, bm25_topics, backlink_list)
        topics = [cid for cid, _ in fused_topics.most_common(N_TOPICS) if cid in concepts]

    # 4) links: sections cited by the chosen topic pages
    linked = []
    if use_links:
        cited = {s for tp in topics for s in out_links[tp] if not is_wiki(s)}
        linked = sorted(cited, key=lambda s: bm25.get(s, 0), reverse=True)[:15]
    timing["okf_ms"] = (time.perf_counter() - t) * 1000

    # 5) fuse the section lists
    fused = rrf(*vector_lists, bm25_sections, linked)
    if papers:
        for cid in fused:
            if paper_of(cid) in papers:
                fused[cid] *= PAPER_BOOST
    sources = {"vector": {c for lst in vector_lists for c in lst}, "bm25": set(bm25_sections), "link": set(linked)}
    sections, per_paper = [], Counter()
    for cid, s in fused.most_common():
        if not papers and per_paper[paper_of(cid)] >= PER_PAPER:
            continue
        found_by = "+".join(name for name, ids in sources.items() if cid in ids)
        sections.append((cid, found_by, round(s * 1000, 1)))
        per_paper[paper_of(cid)] += 1
        if len(sections) == N_SECTIONS:
            break
    return {"question": question, "parts": parts, "papers": papers, "topics": topics, "sections": sections, "timing": timing}


def show_retrieval(r):
    print(f"paper named: {r['papers'] or '-'}" + (f" | parts: {r['parts']}" if r["parts"] else ""))
    for cid in r["topics"]:
        print(f"   topic   {cid:50} {concepts[cid]['title']}")
    for cid, who, s in r["sections"]:
        print(f"   section {who:18} {s:5.1f}  {cid}")
    print("   timing: " + " | ".join(f"{k} {v:.0f}" for k, v in r["timing"].items()))


show_retrieval(retrieve("What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?"))

paper named: - | parts: ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
   topic   topics/rlhf                                        Reinforcement learning from human feedback (RLHF)
   topic   topics/instructgpt                                 InstructGPT
   section vector+bm25+link    65.0  papers/gpt-4-technical-report/5-limitations
   section vector+bm25+link    62.0  papers/gpt-4-technical-report/b-impact-of-rlhf-on-capability
   section vector+bm25         60.1  papers/sparks-of-agi/10-2-on-the-path-to-more-general-artificial-intelligenc-part-2
   section vector+bm25+link    47.1  papers/llm-survey/5-2-5-remarks-on-sft-and-rlhf-part-1
   section vector+bm25+link    46.7  papers/llm-survey/5-2-3-reinforcement-learning-from-human-feedback-part-1
   timing: vector_ms 1156 | okf_ms 1198


## 7. Context, LLM and `ask()`

The same rendering, system prompt and LLM call as `02_OKF_query`. Topic pages go first (the overview), then the sections, until the 18,000-character budget is used.

`ask()` returns the same keys as the other three notebooks, including:
- `generation_ms`: wall-clock time of the LLM call
- `retrieval_context`: the exact blocks the model saw, in order. DeepEval grades from these.

In [8]:
def render(cid):
    c = concepts[cid]
    body = c["body"].split("\n## Links")[0].split("\n## Related topics")[0]
    body = re.sub(r"\n{3,}", "\n\n", LINK_RE.sub(r"\1", body)).strip()[:MAX_CHARS_PER_CONCEPT]
    if c["type"] == "Paper Section":
        first, last = c["pages"] or [None, None]
        where = f"{c['paper']}, " + (f"p. {first}" if first == last else f"pp. {first}-{last}")
        return f"### [{cid}] ({where})\n{body}"
    return f"### [{cid}] (curated {c['type']} page)\n{body}"


def build_context(r, budget=CONTEXT_CHARS):
    blocks, used, size = [], [], 0
    for cid in r["topics"] + [cid for cid, _, _ in r["sections"]]:
        block = render(cid)
        if size + len(block) > budget:
            continue
        blocks.append(block)
        used.append(cid)
        size += len(block)
    return "\n\n".join(blocks), used, blocks


SYSTEM_PROMPT = '''You answer questions about five AI research papers using ONLY the knowledge below.
Each block starts with ### [concept-id]. Rules:
- Use only facts stated in the knowledge blocks. Do not add outside knowledge.
- After each claim, cite the concept id(s) you used in square brackets, e.g. [topics/calibration] or [papers/helm/4-4-calibration-and-uncertainty].
- Keep numbers, model names and percentages exactly as written.
- If the knowledge does not contain the answer, say: "The bundle does not contain this." and stop.
- Be concise: a short paragraph or a few bullets.'''


def ask_llm(question, context):
    start = time.perf_counter()
    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": MODEL_NAME, "stream": False,
              "options": {"temperature": 0, "num_ctx": NUM_CTX, "num_predict": MAX_ANSWER_TOKENS},
              "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                           {"role": "user", "content": f"Knowledge:\n\n{context}\n\n---\n\nQuestion: {question}"}]},
        timeout=600,
    )
    response.raise_for_status()
    data = response.json()
    answer = re.sub(r"<think>.*?</think>", "", data["message"]["content"], flags=re.S).strip()
    return answer, {"prompt_tokens": data.get("prompt_eval_count"), "answer_tokens": data.get("eval_count"),
                    "generation_ms": round((time.perf_counter() - start) * 1000, 1)}


def sources_for(answer, used):
    lines = []
    for cid in used:
        if cid in answer:
            c = concepts[cid]
            if c["type"] == "Paper Section":
                first, last = c["pages"] or [None, None]
                lines.append(f"{cid}  ->  {c['paper']}, {c['title']}, " + (f"p. {first}" if first == last else f"pp. {first}-{last}"))
            else:
                lines.append(f"{cid}  ->  curated {c['type']} page: {c['title']}")
    return lines


def ask(question, show_context=False, **retrieve_kwargs):
    t0 = time.perf_counter()
    r = retrieve(question, **retrieve_kwargs)
    context, used, blocks = build_context(r)
    retrieval_ms = (time.perf_counter() - t0) * 1000
    answer, stats = ask_llm(question, context)

    print(f"Q: {question}\n")
    show_retrieval(r)
    if show_context:
        print("\n--- context ---\n" + context + "\n--- end context ---")
    print("\nanswer:\n" + answer)
    print("\nsources cited:")
    for line in sources_for(answer, used) or ["(none - check the answer!)"]:
        print("   " + line)
    print(f"\ntiming: retrieval {retrieval_ms:.0f} ms | generation {stats['generation_ms']:.0f} ms | "
          f"tokens in/out {stats['prompt_tokens']}/{stats['answer_tokens']} | context {len(context)} chars")
    return {"question": question, "answer": answer, "retrieved": r, "used": used, "retrieval_context": blocks,
            "retrieval_ms": round(retrieval_ms, 1), **stats}


start = time.perf_counter()
_ = ask_llm("Say OK.", "### [warmup]\nOK")
print(f"warm-up took {time.perf_counter() - start:.1f}s")

warm-up took 10.5s


## 8. What does each part add? (retrieval only, no LLM)

The same check as section 12 of the graph hybrid: for each question, how many of the chosen **sections** come from the section that holds the answer.
- **vector only**: Qdrant chunks mapped to sections
- **OKF only**: BM25 + links, close to `02_OKF_query`
- **hybrid**: everything

In [9]:
def chunks_in(doc, pattern):
    return {c["chunk_id"] for c in chunks if c["document"] == doc and re.search(pattern, " > ".join(c["section_path"]))}


GOLD_CHUNKS = {
    "How does HELM measure calibration of language models?": chunks_in("Holistic_Eval", r"4\.4 Calibration|C\.2 Calibration"),
    "What did the GPT-4 technical report say about predictable scaling?": chunks_in("GPT-4", r"^3 Predictable"),
    "What is homogenization in foundation models and why is it risky?": chunks_in("Foundation_Models", r"1\.1 Emergence and homogenization|5\.6\.1 Homogenization"),
    "How did the Sparks of AGI paper test GPT-4 on theory of mind?": chunks_in("AGI", r"^6\.1"),
    "What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?": {"GPT-4_0036"} | chunks_in("LLMs", r"5\.2\.3"),
    "Which model was most accurate on TruthfulQA in HELM?": {"Holistic_Eval_0038"},
    "What score did GPT-4 get on the bar exam?": {c["chunk_id"] for c in chunks if c["document"] == "GPT-4"
                                                   and re.search(r"bar exam", c["text"], re.I) and re.search(r"top 10%|298|90th", c["text"])},
}
GOLD = {q: {section_of_chunk[c] for c in ids if c in section_of_chunk} for q, ids in GOLD_CHUNKS.items()}   # chunks -> sections
MODES = {"vector only": dict(use_vectors=True, use_links=False, split=False, use_topics=False),
         "OKF only": dict(use_vectors=False, split=False),
         "hybrid": dict()}

rows = []
for question, gold in GOLD.items():
    row = {"question": question[:55]}
    for mode, kwargs in MODES.items():
        row[mode] = len({cid for cid, _, _ in retrieve(question, **kwargs)["sections"]} & gold)
    rows.append(row)
check = pd.DataFrame(rows)
display(check)
print("questions with at least one section from the right place:", {m: f"{(check[m] > 0).sum()}/{len(check)}" for m in MODES})

,question,vector only,OKF only,hybrid
0,How does HELM measure calibration of language ...,1,2,0
1,What did the GPT-4 technical report say about ...,3,3,3
2,What is homogenization in foundation models an...,2,2,2
3,How did the Sparks of AGI paper test GPT-4 on ...,4,4,5
4,"What are the three steps of RLHF, and how did ...",1,2,2
5,Which model was most accurate on TruthfulQA in...,0,0,0
6,What score did GPT-4 get on the bar exam?,2,2,2


questions with at least one section from the right place: {'vector only': '6/7', 'OKF only': '6/7', 'hybrid': '5/7'}


## 9. Ask the seven questions

In [10]:
r1 = ask("How does HELM measure calibration of language models?")

Q: How does HELM measure calibration of language models?

paper named: ['helm']
   topic   papers/helm/overview                               Holistic Evaluation of Language Models (HELM)
   topic   topics/helm-scenarios-and-metrics                  HELM scenarios and metrics
   section vector+bm25+link    72.6  papers/helm/1-1-helm-part-1
   section vector+bm25+link    67.8  papers/helm/1-2-empirical-findings-part-1
   section bm25+link           46.9  papers/helm/4-2-selection
   section bm25+link           46.2  papers/helm/4-1-taxonomy-part-1
   section bm25+link           45.2  papers/helm/abstract
   timing: vector_ms 408 | okf_ms 462

answer:
HELM measures calibration by assessing how well a model's predicted confidence levels align with its actual performance, particularly in specific scenarios. For example, on HellaSwag, improving accuracy worsens calibration, indicating a trade-off between accuracy and calibration. On OpenBookQA, better accuracy improves calibration. These re

In [11]:
r2 = ask("What did the GPT-4 technical report say about predictable scaling?")

Q: What did the GPT-4 technical report say about predictable scaling?

paper named: ['gpt-4-technical-report']
   topic   topics/predictable-scaling                         Predictable scaling
   topic   papers/gpt-4-technical-report/overview             GPT-4 Technical Report
   section vector+bm25+link    73.8  papers/gpt-4-technical-report/3-predictable-scaling
   section vector+bm25+link    71.8  papers/gpt-4-technical-report/3-1-loss-prediction
   section vector+bm25+link    71.4  papers/gpt-4-technical-report/3-2-scaling-of-capabilities-on-humaneval
   section vector+bm25+link    65.5  papers/gpt-4-technical-report/abstract
   section vector+bm25         47.3  papers/gpt-4-technical-report/7-conclusion-part-1
   timing: vector_ms 1049 | okf_ms 396

answer:
The GPT-4 technical report states that a core component of the project was developing infrastructure and optimization methods that behave predictably across a wide range of model scales [topics/predictable-scaling]. This predic

In [12]:
r3 = ask("What is homogenization in foundation models and why is it risky?")

Q: What is homogenization in foundation models and why is it risky?

paper named: -
   topic   topics/homogenization                              Homogenization
   topic   topics/foundation-model                            Foundation model
   section vector+bm25+link    49.2  papers/foundation-models/1-1-emergence-and-homogenization-part-3
   section vector+bm25+link    48.1  papers/foundation-models/1-1-emergence-and-homogenization-part-1
   section bm25                14.3  papers/gpt-4-technical-report/system-card/2-9-potential-for-risky-emergent-behaviors
   section link                14.1  papers/helm/abstract
   section bm25                13.7  papers/gpt-4-technical-report/system-card/abstract
   timing: vector_ms 1174 | okf_ms 426

answer:
Homogenization in foundation models refers to the consolidation of methodologies for building machine learning systems across a wide range of applications, such that one model (e.g., GPT-3) can be adapted to many downstream tasks (1.1 Emerg

In [13]:
r4 = ask("How did the Sparks of AGI paper test GPT-4 on theory of mind?")

Q: How did the Sparks of AGI paper test GPT-4 on theory of mind?

paper named: ['sparks-of-agi']
   topic   topics/theory-of-mind                              Theory of mind
   topic   papers/sparks-of-agi/overview                      Sparks of Artificial General Intelligence: Early experiments with GPT-4
   section vector+bm25+link    73.0  papers/sparks-of-agi/6-1-1-testing-specific-aspects-of-theory-of-mind
   section vector+bm25+link    71.8  papers/sparks-of-agi/6-1-2-testing-theory-of-mind-in-realistic-scenarios-part-1
   section bm25+link           48.4  papers/sparks-of-agi/6-1-3-discussion
   section vector+bm25         47.7  papers/sparks-of-agi/6-1-2-testing-theory-of-mind-in-realistic-scenarios-part-2
   section bm25+link           46.9  papers/sparks-of-agi/6-1-understanding-humans-theory-of-mind
   timing: vector_ms 1138 | okf_ms 405

answer:
The Sparks of AGI paper tested GPT-4 on theory of mind through a series of designed scenarios that required reasoning about the me

In [14]:
r5 = ask("Which papers discuss hallucination, and what does each one say about it?")

Q: Which papers discuss hallucination, and what does each one say about it?

paper named: -
   topic   topics/hallucination                               Hallucination
   topic   topics/truthfulqa                                  TruthfulQA
   section vector+bm25+link    48.4  papers/llm-survey/9-7-3-hallucination-detection
   section vector+bm25+link    48.2  papers/llm-survey/9-7-2-source-of-hallucination-part-1
   section bm25+link           29.0  papers/sparks-of-agi/10-2-on-the-path-to-more-general-artificial-intelligenc-part-1
   section bm25+link           27.6  papers/gpt-4-technical-report/system-card/2-2-hallucinations
   section bm25                15.9  papers/sparks-of-agi/1-2-organization-of-our-demonstration-part-1
   timing: vector_ms 1146 | okf_ms 441

answer:
- **GPT-4 Technical Report and System Card**: Defines hallucination as producing "nonsensical or untruthful" content, distinguishing closed-domain (inconsistencies with input) and open-domain (false facts about t

In [15]:
r6 = ask("What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?")

Q: What are the three steps of RLHF, and how did RLHF affect GPT-4's calibration?

paper named: - | parts: ['What are the three steps of RLHF', "how did RLHF affect GPT-4's calibration"]
   topic   topics/rlhf                                        Reinforcement learning from human feedback (RLHF)
   topic   topics/instructgpt                                 InstructGPT
   section vector+bm25+link    65.0  papers/gpt-4-technical-report/5-limitations
   section vector+bm25+link    62.0  papers/gpt-4-technical-report/b-impact-of-rlhf-on-capability
   section vector+bm25         60.1  papers/sparks-of-agi/10-2-on-the-path-to-more-general-artificial-intelligenc-part-2
   section vector+bm25+link    47.1  papers/llm-survey/5-2-5-remarks-on-sft-and-rlhf-part-1
   section vector+bm25+link    46.7  papers/llm-survey/5-2-3-reinforcement-learning-from-human-feedback-part-1
   timing: vector_ms 1914 | okf_ms 1212

answer:
The three steps of RLHF are:  
1. Supervised fine-tuning on human-written p

In [16]:
r7 = ask("How many parameters does Gemini Ultra have?")

Q: How many parameters does Gemini Ultra have?

paper named: -
   topic   topics/parameter-efficient-fine-tuning             Parameter-efficient fine-tuning (PEFT)
   topic   topics/gpt-3                                       GPT-3
   section vector+bm25+link    48.2  papers/llm-survey/2-2-technical-evolution-of-gpt-series-models-part-3
   section vector+bm25+link    48.1  papers/foundation-models/4-3-1-methods-for-foundation-model-adaptation-part-1
   section vector+bm25+link    45.7  papers/foundation-models/1-introduction
   section vector+bm25         30.8  papers/helm/7-adaptation-via-prompting-part-1
   section vector+bm25         28.7  papers/sparks-of-agi/8-1-warm-up-with-two-basic-examples
   timing: vector_ms 1067 | okf_ms 407

answer:
The bundle does not contain this.

sources cited:
   (none - check the answer!)

timing: retrieval 1477 ms | generation 4317 ms | tokens in/out 4461/8 | context 14573 chars


In [17]:
summary = pd.DataFrame([{k: r[k] for k in ("question", "retrieval_ms", "generation_ms", "prompt_tokens", "answer_tokens")}
                        for r in (r1, r2, r3, r4, r5, r6, r7)])
summary["question"] = summary["question"].str[:60]
display(summary)

,question,retrieval_ms,generation_ms,prompt_tokens,answer_tokens
0,How does HELM measure calibration of language ...,871.4,7265.2,4616,152
1,What did the GPT-4 technical report say about ...,1447.9,10722.1,4260,309
2,What is homogenization in foundation models an...,1609.4,11170.2,4081,338
3,How did the Sparks of AGI paper test GPT-4 on ...,1543.9,13014.8,3752,435
4,"Which papers discuss hallucination, and what d...",1600.8,13879.0,3958,472
5,"What are the three steps of RLHF, and how did ...",3145.0,7483.0,4324,154
6,How many parameters does Gemini Ultra have?,1476.9,4317.2,4461,8


## Save results for the comparison

This runs every question in `data/eval/eval_questions.jsonl` through `ask()` (quietly) and saves one line per question to `artifacts/results/okf_hybrid.jsonl`. Each line holds:
- the **answer**
- **`retrieval_context`**: the exact blocks the model saw, in order
- **timing**: `retrieval_ms`, `generation_ms`
- **tokens**: `prompt_tokens`, `answer_tokens`

`08_evaluate_methods.ipynb` grades all four methods from these files with DeepEval, using one judge, so every method is judged the same way. Re-run this cell whenever you change the method or the questions.

In [18]:
import contextlib
import io

METHOD = "okf_hybrid"
EVAL_PATH = PROJECT_ROOT / "data" / "eval" / "eval_questions.jsonl"
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results" / f"{METHOD}.jsonl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

# a hybrid result only counts if its vector parts are really on (otherwise this would save a keyword-only run)
assert QDRANT_OK and USE_VECTORS and TOPICS_IN_QDRANT, "Qdrant isn't connected or USE_VECTORS is off - fix that before saving results"
print(f"switches: USE_VECTORS={USE_VECTORS} USE_LINKS={USE_LINKS} SPLIT_QUESTIONS={SPLIT_QUESTIONS}")

eval_questions = [json.loads(line) for line in EVAL_PATH.open(encoding="utf-8") if line.strip()]
print(f"{len(eval_questions)} questions -> {RESULTS_PATH}")


def ask_quietly(question, attempts=2):
    # one retry, in case Ollama or Qdrant hiccups
    for attempt in range(attempts):
        try:
            with contextlib.redirect_stdout(io.StringIO()):          # ask() prints a lot; keep this cell readable
                return ask(question)
        except Exception as exc:
            if attempt == attempts - 1:
                raise
            print(f"   failed ({type(exc).__name__}: {str(exc)[:80]}), retrying in 10 s")
            time.sleep(10)


# written line by line: if it stops halfway, the answers so far are kept (08 shows the rest as missing)
with RESULTS_PATH.open("w", encoding="utf-8") as f:
    for n, item in enumerate(eval_questions, start=1):
        r = ask_quietly(item["question"])
        row = {"id": item["id"], "method": METHOD, "question": item["question"], "answer": r["answer"],
               "retrieval_context": r["retrieval_context"], "retrieval_ms": r["retrieval_ms"],
               "generation_ms": r["generation_ms"], "prompt_tokens": r["prompt_tokens"],
               "answer_tokens": r["answer_tokens"]}
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{n:2}/{len(eval_questions)}] {item['id']:5} retrieval {r['retrieval_ms']:6.0f} ms | "
              f"generation {r['generation_ms']:6.0f} ms | {r['answer'][:60]!r}")
print(f"saved {len(eval_questions)} results to {RESULTS_PATH.name}")

switches: USE_VECTORS=True USE_LINKS=True SPLIT_QUESTIONS=True
30 questions -> c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\artifacts\results\okf_hybrid.jsonl
[ 1/30] q01   retrieval    900 ms | generation   6068 ms | 'text-davinci-002 was most accurate on TruthfulQA in HELM, wi'
[ 2/30] q02   retrieval   1477 ms | generation   5637 ms | 'GPT-4 scored 298/400 on the Uniform Bar Exam, which correspo'
[ 3/30] q03   retrieval   1565 ms | generation   7932 ms | 'GPT-4 was accurate only 58% on simple arithmetic expressions'
[ 4/30] q04   retrieval   3595 ms | generation   6621 ms | "GPT-4's safety mitigations reduced responses to disallowed c"
[ 5/30] q05   retrieval   1592 ms | generation   6442 ms | 'According to the LLM survey, the pre-training data mixture f'
[ 6/30] q06   retrieval   1842 ms | generation   7915 ms | "In the LLM survey's quantization experiments, LLaMA (7B) nee"
[ 7/30] q07   retrieval   1551 ms | generation   7725 ms | "HELM measures calibration by assessing how well a

## Experiments worth trying

1. **One part at a time:** `ask(q, use_links=False, use_topics=False)` is vector search + BM25 over sections. `ask(q, use_vectors=False)` is close to `02_OKF_query`.
2. **No splitting:** `ask(q6, split=False)` on the RLHF question.
3. **See what the model saw:** `ask(q, show_context=True)`.

Things to know:
- Sections are long (up to 4,500 characters), so only 3–4 fit after the topic pages. The graph hybrid fits ~8 chunks plus facts. Same budget, different granularity.
- The vector part is identical to the graph hybrid's (`okf_chunks`). Only what's built on top differs.